In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
"""
DDIN v5 — Vaikharī: Dhātu Activation → Symbolic Output
=========================================================

Building on exp04 (Dhātu extraction).

The Vaikharī layer is the output interface:

    Dynamics → Dhātu activation pattern → symbolic label

This closes the loop:
    Parā      → pre-symbolic ground          (v1/v2)
    Paśyantī  → coherent attractor states    (v3)
    Madhyamā  → Dhātu subgraphs              (v4)
    Vaikharī  → symbolic output              ← THIS

---

What we're proving in v5:

  1. Different input modalities produce SEPARABLE Dhātu activation codes
  2. The Dhātu code is compressible → a semantic fingerprint
  3. A lightweight decoder can map Dhātu states → category labels
  4. The mapping is interpretable: each Dhātu role maps to a semantic category

This is NOT supervised classification.
We use the Dhātu structure from v4 as the encoding, then show
that the encoding ALREADY contains the semantic distinctions —
no labels needed first.

---

Sanskrit grounding (important):

  The Vaikharī is the level of SPEECH — audible, communicable output.
  In DDIN terms: the first layer that can be "heard" externally.

  The four levels of speech:
    Parā      → undifferentiated potential (physics layer)
    Paśyantī  → simultaneous holistic awareness (attractor layer)
    Madhyamā  → structured thought (Dhātu subgraph layer)
    Vaikharī  → spoken word (symbolic output layer)   ← here

"""

import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from scipy.signal import find_peaks

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("="*60)


# ────────────────────────────────────────────────────────────────
# 1.  DDIN MODEL (identical to v4 — do NOT change)
# ────────────────────────────────────────────────────────────────

class HeterogeneousLiquidSystem(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.W     = nn.Parameter(torch.randn(dim, dim) * 0.05)
        self.alpha = nn.Parameter(torch.rand(dim) * 0.6 + 0.1)
        self.beta  = nn.Parameter(torch.rand(dim) * 0.4 + 0.05)

    def forward(self, x, u, dt=0.1):
        dx = -self.alpha * x + torch.tanh(x @ self.W) + self.beta * u
        return x + dt * dx


# ────────────────────────────────────────────────────────────────
# 2.  INPUT GENERATOR (v4 set + 4 new modes for richer labeling)
# ────────────────────────────────────────────────────────────────

def generate_input(seq_len, dim, mode="sine"):
    t = torch.linspace(0, 10, seq_len).to(device)

    if mode == "sine":
        freqs = torch.linspace(0.8, 1.2, dim).to(device)
        return torch.sin(t.unsqueeze(1) * freqs.unsqueeze(0))

    elif mode == "cosine":
        freqs = torch.linspace(0.8, 1.2, dim).to(device)
        return torch.cos(t.unsqueeze(1) * freqs.unsqueeze(0))

    elif mode == "constant":
        levels = torch.linspace(0.2, 0.8, dim).to(device)
        return levels.unsqueeze(0).repeat(seq_len, 1)

    elif mode == "impulse":
        u = torch.zeros(seq_len, dim).to(device)
        for i in range(dim):
            start = 5 + (i % 10) * 2
            u[start:start + 3, i] = 1.0
        return u

    elif mode == "noise":
        return torch.randn(seq_len, dim).to(device) * 0.3

    elif mode == "ramp":
        t_norm = t / t.max()
        return t_norm.unsqueeze(1) * torch.linspace(0.1, 1.0, dim).to(device).unsqueeze(0)

    elif mode == "anti_sine":
        freqs = torch.linspace(0.8, 1.2, dim).to(device)
        return -torch.sin(t.unsqueeze(1) * freqs.unsqueeze(0))

    elif mode == "burst":
        u = torch.zeros(seq_len, dim).to(device)
        for b in range(0, seq_len, 30):
            u[b:b + 5, :] = 1.0
        return u

    elif mode == "slow_sine":
        freqs = torch.linspace(0.2, 0.4, dim).to(device)
        return torch.sin(t.unsqueeze(1) * freqs.unsqueeze(0))

    elif mode == "fast_sine":
        freqs = torch.linspace(2.0, 3.0, dim).to(device)
        return torch.sin(t.unsqueeze(1) * freqs.unsqueeze(0))

    raise ValueError(f"Unknown mode: {mode}")


# ────────────────────────────────────────────────────────────────
# 3.  TRAIN (same two-phase protocol as v3/v4)
# ────────────────────────────────────────────────────────────────

dim     = 64
seq_len = 200
EPOCHS  = 400

model     = HeterogeneousLiquidSystem(dim=dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

loss_history    = []
synapse_history = []

print("Training DDIN (same protocol as v4)...")
print("="*50)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    x = torch.zeros(1, dim).to(device)
    u = generate_input(seq_len, dim, mode="sine")

    pred_loss = smooth_loss = energy_loss = 0.0
    for t_step in range(seq_len - 1):
        x_next   = model(x,      u[t_step])
        x_future = model(x_next, u[t_step + 1])
        pred_loss   += torch.mean((x_future - x_next.detach()) ** 2)
        energy_loss += torch.mean(torch.abs(x))
        smooth_loss += torch.mean((x_next - x) ** 2)
        x = x_next

    if epoch < 200:
        loss = pred_loss + 0.1 * smooth_loss
    else:
        loss = pred_loss + 0.1 * smooth_loss + 0.02 * energy_loss + \
               0.05 * torch.sum(torch.abs(model.W))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if epoch >= 200 and epoch % 20 == 0:
        with torch.no_grad():
            model.W[torch.abs(model.W) < 0.02] = 0.0

    active = torch.count_nonzero(torch.abs(model.W) > 0.01).item()
    synapse_history.append(active)
    loss_history.append(loss.item())

    if epoch % 50 == 0:
        phase = "Learning" if epoch < 200 else "Pruning "
        print(f"  Epoch {epoch:3d} ({phase}) | loss={loss.item():.4f} | synapses={active}")

sparsity = 1.0 - synapse_history[-1] / (dim * dim)
print(f"\nFinal sparsity: {sparsity:.1%}   |   Synapses: {synapse_history[-1]}")


# ────────────────────────────────────────────────────────────────
# 4.  RE-DO DHĀTU CLUSTERING (same as v4)
# ────────────────────────────────────────────────────────────────

TRAINING_MODES = ["sine", "cosine", "constant", "impulse", "noise", "ramp"]
N_DHATU = 5

def collect_trajectory(mode, steps=300):
    x = torch.zeros(1, dim).to(device)
    u = generate_input(steps, dim, mode=mode)
    states = []
    with torch.no_grad():
        for t_step in range(steps):
            x = model(x, u[t_step])
            states.append(x.cpu().numpy())
    return np.array(states).squeeze()

print("\nCollecting base trajectories for Dhātu clustering...")
trajs = {m: collect_trajectory(m) for m in TRAINING_MODES}

mean_acts = {m: np.mean(np.abs(trajs[m]), axis=0) for m in TRAINING_MODES}
var_acts  = {m: np.var(trajs[m],          axis=0) for m in TRAINING_MODES}

MODE_PAIRS = [
    ("sine", "cosine"), ("sine", "constant"), ("sine", "impulse"),
    ("cosine", "impulse"), ("constant", "noise"),
]
mean_diff = np.mean(np.stack(
    [np.abs(mean_acts[m1] - mean_acts[m2]) for m1, m2 in MODE_PAIRS], axis=0
), axis=0)

W = model.W.detach().cpu().numpy()
hub_score = np.sum(np.abs(W), axis=1) + np.sum(np.abs(W), axis=0)

def autocorr_peak(signal, lag_start=5, lag_end=60):
    signal = signal - np.mean(signal)
    if np.std(signal) < 1e-6:
        return 0.0
    ac = np.correlate(signal, signal, mode='full')
    ac = ac[len(ac)//2:]
    ac /= (ac[0] + 1e-8)
    segment = ac[lag_start:lag_end]
    if len(segment) == 0:
        return 0.0
    peaks, _ = find_peaks(segment, height=0.2)
    return float(ac[peaks[0] + lag_start]) if len(peaks) > 0 else 0.0

periodicity = np.array([autocorr_peak(trajs["sine"][:, i]) for i in range(dim)])

F = np.column_stack([
    mean_acts["sine"], mean_acts["cosine"], mean_acts["impulse"],
    var_acts["sine"], var_acts["impulse"],
    mean_diff, hub_score, periodicity,
])

scaler  = StandardScaler()
F_norm  = scaler.fit_transform(F)
kmeans  = KMeans(n_clusters=N_DHATU, random_state=42, n_init=30, max_iter=500)
cluster_labels = kmeans.fit_predict(F_norm)

print(f"Dhātu cluster sizes: {[int(np.sum(cluster_labels==k)) for k in range(N_DHATU)]}")


# ────────────────────────────────────────────────────────────────
# 5.  VAIKHARĪ CORE: DHĀTU ACTIVATION CODES
# ────────────────────────────────────────────────────────────────
#
#  For each input mode, we compute the DHĀTU ACTIVATION CODE:
#    A 5-dimensional vector: code[k] = mean |activity| of neurons in Dhātu k
#
#  This is the symbolic interface — the Vaikharī projection.
#  It maps the continuous neural trajectory to a discrete semantic fingerprint.
# ────────────────────────────────────────────────────────────────

ALL_MODES = [
    "sine", "cosine", "constant", "impulse", "noise", "ramp",
    "anti_sine", "burst", "slow_sine", "fast_sine",
]

# Semantic category labels (human interpretation)
MODE_CATEGORIES = {
    "sine"      : "OSCILLATION",
    "cosine"    : "OSCILLATION",
    "slow_sine" : "SLOW_WAVE",
    "fast_sine" : "FAST_WAVE",
    "anti_sine" : "INVERSION",
    "constant"  : "STEADY_STATE",
    "ramp"      : "ACCUMULATION",
    "impulse"   : "EVENT",
    "burst"     : "BURST",
    "noise"     : "STOCHASTIC",
}

def compute_dhatu_code(traj, cluster_labels, n_dhatu=N_DHATU):
    """
    Project trajectory to Dhātu activation space.

    For each Dhātu k, compute:
      code[k] = mean |activity| across all neurons in that Dhātu,
                averaged over time

    Returns: (n_dhatu,) vector = semantic fingerprint of this input
    """
    code = np.zeros(n_dhatu)
    for k in range(n_dhatu):
        mask = cluster_labels == k
        if mask.any():
            code[k] = float(np.mean(np.abs(traj[:, mask])))
    return code


# Compute codes for all modes (multiple realizations for robustness)
N_REALIZATIONS = 8   # run each mode N times to get a distribution

print("\nComputing Dhātu activation codes for all modalities...")
all_codes  = []   # (N_modes × N_realizations, N_DHATU)
all_labels = []   # category label per code
all_modes  = []   # raw mode name

for mode in ALL_MODES:
    cat = MODE_CATEGORIES[mode]
    for r in range(N_REALIZATIONS):
        # slight random perturbation to get distribution
        torch.manual_seed(r * 137)
        traj = collect_trajectory(mode)
        code = compute_dhatu_code(traj, cluster_labels)
        all_codes.append(code)
        all_labels.append(cat)
        all_modes.append(mode)

all_codes  = np.array(all_codes)    # (80, 5)
all_labels = np.array(all_labels)   # (80,)
all_modes  = np.array(all_modes)    # (80,)

print(f"Code matrix: {all_codes.shape}  (samples × Dhātu)")
print(f"Categories : {sorted(set(all_labels))}")


# ────────────────────────────────────────────────────────────────
# 6.  SEPARABILITY ANALYSIS
# ────────────────────────────────────────────────────────────────

# 6a. PCA of Dhātu code space
pca_v = PCA(n_components=2)
codes_2d_pca = pca_v.fit_transform(all_codes)

# 6b. Mean code per modality (prototype vectors)
unique_modes = sorted(set(all_modes))
prototypes = {}
for mode in unique_modes:
    mask = all_modes == mode
    prototypes[mode] = np.mean(all_codes[mask], axis=0)

# 6c. Category separability: mean intra-class vs inter-class distance
from itertools import combinations

categories = sorted(set(all_labels))
intra_dists = []
inter_dists = []

for cat in categories:
    subset = all_codes[all_labels == cat]
    if len(subset) < 2:
        continue
    for i in range(len(subset)):
        for j in range(i + 1, len(subset)):
            intra_dists.append(np.linalg.norm(subset[i] - subset[j]))

for cat1, cat2 in combinations(categories, 2):
    s1 = all_codes[all_labels == cat1]
    s2 = all_codes[all_labels == cat2]
    for c1 in s1:
        for c2 in s2:
            inter_dists.append(np.linalg.norm(c1 - c2))

mean_intra = np.mean(intra_dists)
mean_inter = np.mean(inter_dists)
separation_ratio = mean_inter / (mean_intra + 1e-8)

print(f"\nSeparability:")
print(f"  Mean intra-class distance : {mean_intra:.4f}")
print(f"  Mean inter-class distance : {mean_inter:.4f}")
print(f"  Separation ratio          : {separation_ratio:.2f}x")
print(f"  (ratio > 2.0 = good separability, > 5.0 = excellent)")


# ────────────────────────────────────────────────────────────────
# 7.  VAIKHARĪ DECODER: LIGHTWEIGHT MLP
#     Maps Dhātu code → category label
# ────────────────────────────────────────────────────────────────

class VaikhariDecoder(nn.Module):
    """
    The Vaikharī projection head.

    Input  : Dhātu activation code (N_DHATU,)
    Output : category logits

    This is intentionally SMALL:
    The semantic work is done by the dynamics.
    The decoder only needs to read out the code.
    """
    def __init__(self, n_dhatu, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_dhatu, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, n_classes),
        )

    def forward(self, x):
        return self.net(x)


le = LabelEncoder()
label_ids = le.fit_transform(all_labels)
n_classes = len(le.classes_)

print(f"\nTraining Vaikharī decoder...")
print(f"  Input: {N_DHATU}-dim Dhātu code")
print(f"  Output: {n_classes} categories → {list(le.classes_)}")

# Train/val split
np.random.seed(42)
idx = np.random.permutation(len(all_codes))
split = int(0.75 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

X_train = torch.FloatTensor(all_codes[train_idx])
y_train = torch.LongTensor(label_ids[train_idx])
X_val   = torch.FloatTensor(all_codes[val_idx])
y_val   = torch.LongTensor(label_ids[val_idx])

decoder   = VaikhariDecoder(N_DHATU, n_classes)
opt_dec   = torch.optim.Adam(decoder.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

dec_losses = []
dec_accs   = []

for epoch in range(300):
    decoder.train()
    opt_dec.zero_grad()
    logits = decoder(X_train)
    loss   = criterion(logits, y_train)
    loss.backward()
    opt_dec.step()

    with torch.no_grad():
        decoder.eval()
        val_logits = decoder(X_val)
        val_pred   = torch.argmax(val_logits, dim=1)
        val_acc    = (val_pred == y_val).float().mean().item()

    dec_losses.append(loss.item())
    dec_accs.append(val_acc)

decoder.eval()
with torch.no_grad():
    final_pred = torch.argmax(decoder(X_val), dim=1).numpy()

final_acc = np.mean(final_pred == label_ids[val_idx])
print(f"\nVaikharī decoder — final validation accuracy: {final_acc:.1%}")
print(f"(chance = {1/n_classes:.1%})")

# Confusion matrix — force full (n_classes × n_classes) shape so the
# rendering loop never goes out of bounds even when some classes are
# absent from the validation split.
cm = confusion_matrix(label_ids[val_idx], final_pred,
                      labels=list(range(n_classes)))


# ────────────────────────────────────────────────────────────────
# 8.  PROTOTYPE INTERPRETATION TABLE
# ────────────────────────────────────────────────────────────────

print("\nDhātu Activation Prototypes (mean code per mode):")
dhatu_header = "  ".join(f"   D{k}" for k in range(N_DHATU))
print(f"  {'Mode':15s} | Cat         | {dhatu_header}")
print("  " + "-"*75)
for mode in unique_modes:
    p   = prototypes[mode]
    cat = MODE_CATEGORIES[mode]
    vals = "  ".join(f"{v:5.3f}" for v in p)
    print(f"  {mode:15s} | {cat:11s} | {vals}")


# ────────────────────────────────────────────────────────────────
# 9.  END-TO-END PIPELINE TEST
# ────────────────────────────────────────────────────────────────
#
#  Full pipeline: raw input → DDIN dynamics → Dhātu code → label
#

print("\nEnd-to-end pipeline test:")
print("-"*50)
test_cases = [
    ("sine",      "OSCILLATION  (expected)"),
    ("impulse",   "EVENT        (expected)"),
    ("constant",  "STEADY_STATE (expected)"),
    ("burst",     "BURST        (expected)"),
    ("slow_sine", "SLOW_WAVE    (expected)"),
    ("noise",     "STOCHASTIC   (expected)"),
]

for mode, expected in test_cases:
    traj = collect_trajectory(mode)
    code = compute_dhatu_code(traj, cluster_labels)
    code_t = torch.FloatTensor(code).unsqueeze(0)
    with torch.no_grad():
        logit  = decoder(code_t)
        pred_id  = torch.argmax(logit, dim=1).item()
        pred_cat = le.inverse_transform([pred_id])[0]
        conf     = torch.softmax(logit, dim=1)[0, pred_id].item()
    status = "✓" if pred_cat == MODE_CATEGORIES[mode] else "✗"
    print(f"  {status} {mode:12s} → [{pred_cat:12s}] (conf={conf:.0%})  | {expected}")


# ────────────────────────────────────────────────────────────────
# 10.  VISUALISATION
# ────────────────────────────────────────────────────────────────

DHATU_COLORS = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff', '#c77dff']
CAT_COLORS = {
    "OSCILLATION" : "#4d96ff",
    "SLOW_WAVE"   : "#6bcb77",
    "FAST_WAVE"   : "#ff6b6b",
    "INVERSION"   : "#c77dff",
    "STEADY_STATE": "#ffd93d",
    "ACCUMULATION": "#ff9944",
    "EVENT"       : "#ffffff",
    "BURST"       : "#ff4444",
    "STOCHASTIC"  : "#888888",
}

fig = plt.figure(figsize=(22, 20))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(
    "DDIN v5  —  Vaikharī Layer: Dhātu Code → Symbolic Output\n"
    "First complete pipeline: dynamics → structure → language",
    fontsize=13, color='white', fontweight='bold', y=0.99
)

def dark_ax(ax, title="", xlabel="", ylabel=""):
    ax.set_facecolor('#111111')
    ax.set_title(title, color='white', fontsize=8.5, pad=4)
    if xlabel: ax.set_xlabel(xlabel, color='#888888', fontsize=7)
    if ylabel: ax.set_ylabel(ylabel, color='#888888', fontsize=7)
    ax.tick_params(colors='#888888', labelsize=6)
    for s in ax.spines.values():
        s.set_color('#333333')


# ── (1) PCA of Dhātu code space ───────────────────────────────
ax1 = fig.add_subplot(4, 4, 1)
dark_ax(ax1, "Dhātu Code Space (PCA)\ninput modes → separable clusters")
for cat in categories:
    mask = all_labels == cat
    ax1.scatter(codes_2d_pca[mask, 0], codes_2d_pca[mask, 1],
                c=CAT_COLORS.get(cat, '#888888'), s=40, alpha=0.75,
                label=cat, zorder=3)
ax1.legend(fontsize=5, facecolor='#222222', labelcolor='white',
           framealpha=0.8, loc='best', ncol=2)

# ── (2) Dhātu activation heatmap (modes × Dhātu) ─────────────
ax2 = fig.add_subplot(4, 4, 2)
dark_ax(ax2, "Prototype Dhātu Codes\n(one row per input mode)")
proto_matrix = np.stack([prototypes[m] for m in unique_modes], axis=0)
im = ax2.imshow(proto_matrix, cmap='hot', aspect='auto', vmin=0)
ax2.set_yticks(range(len(unique_modes)))
ax2.set_yticklabels(unique_modes, fontsize=6, color='#cccccc')
ax2.set_xticks(range(N_DHATU))
ax2.set_xticklabels([f"D{k}" for k in range(N_DHATU)],
                     fontsize=7, color='#cccccc')
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

# ── (3) Decoder training curve ────────────────────────────────
ax3 = fig.add_subplot(4, 4, 3)
dark_ax(ax3, "Vaikharī Decoder Training", "Epoch", "")
ax3.plot(dec_losses, color='#ff9944', lw=0.9, label='loss', alpha=0.9)
ax3b = ax3.twinx()
ax3b.plot(dec_accs,  color='#6bcb77', lw=0.9, label='val acc', alpha=0.9)
ax3b.set_ylabel("Accuracy", color='#6bcb77', fontsize=6)
ax3b.tick_params(colors='#6bcb77', labelsize=5)
ax3b.axhline(1/n_classes, color='gray', lw=0.7, ls='--', alpha=0.5)
ax3.legend(fontsize=6, facecolor='#222222', labelcolor='white', loc='upper right')

# ── (4) Confusion matrix ──────────────────────────────────────
ax4 = fig.add_subplot(4, 4, 4)
dark_ax(ax4, f"Confusion Matrix\n(val acc={final_acc:.1%}, chance={1/n_classes:.0%})")
im4 = ax4.imshow(cm, cmap='Blues', aspect='auto')
class_names = [c[:6] for c in le.classes_]
ax4.set_xticks(range(n_classes))
ax4.set_yticks(range(n_classes))
ax4.set_xticklabels(class_names, rotation=45, ha='right', fontsize=5, color='#cccccc')
ax4.set_yticklabels(class_names, fontsize=5, color='#cccccc')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax4.text(j, i, str(cm[i, j]), ha='center', va='center',
                 fontsize=7, color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax4.set_xlabel("Predicted", color='#888888', fontsize=7)
ax4.set_ylabel("Actual",    color='#888888', fontsize=7)

# ── (5–13) Dhātu code per category (one panel per category) ──
# 4×4 grid: panels 5–16 available; we use 5–13 for categories,
# 14–16 for summary. Supports up to 9 categories safely.
for idx_c, cat in enumerate(sorted(categories)):
    if idx_c >= 9:
        break   # grid supports max 9 category panels (positions 5-13)
    ax = fig.add_subplot(4, 4, 5 + idx_c)
    dark_ax(ax, f"Category: {cat}\nDhātu activation fingerprint",
            "Dhātu", "Mean |act|")
    mask  = all_labels == cat
    mean  = np.mean(all_codes[mask], axis=0)
    std   = np.std(all_codes[mask],  axis=0)
    xpos  = np.arange(N_DHATU)
    ax.bar(xpos, mean, color=DHATU_COLORS, alpha=0.85, width=0.6)
    ax.errorbar(xpos, mean, yerr=std, fmt='none',
                ecolor='white', elinewidth=1, capsize=3)
    ax.set_xticks(xpos)
    ax.set_xticklabels([f"D{k}" for k in range(N_DHATU)], fontsize=7,
                        color='#cccccc')
    for s in ax.spines.values():
        s.set_color(CAT_COLORS.get(cat, '#888888'))
        s.set_linewidth(1.5)
    if idx_c >= 8:
        break   # only 9 panels (5-13) in the 4×4 grid

# ── (14) End-to-end architecture diagram ────────────────────
ax10 = fig.add_subplot(4, 4, 14)
dark_ax(ax10, "DDIN Full Architecture\nParā → Vaikharī pipeline")
ax10.set_xlim(0, 10)
ax10.set_ylim(0, 10)

layers = [
    (5, 8.5, "Parā",     "Physical substrate\ndynamics",      "#555566", 0.9),
    (5, 6.5, "Paśyantī", "Attractor states\n(DDIN v3)",       "#445566", 0.9),
    (5, 4.5, "Madhyamā", "Dhātu subgraphs\n(DDIN v4)",        "#336655", 0.9),
    (5, 2.5, "Vaikharī", "Symbolic output\n← YOU ARE HERE",   "#553322", 0.95),
]
for (x, y, name, desc, bg, alpha) in layers:
    bbox = dict(boxstyle='round,pad=0.5', facecolor=bg, edgecolor='#aaaaaa',
                alpha=alpha, linewidth=1.5)
    ax10.text(x, y, f"{name}\n{desc}", ha='center', va='center',
              fontsize=7, color='white', fontweight='bold', bbox=bbox)
    if y > 2.5:
        ax10.annotate('', xy=(5, y - 0.7), xytext=(5, y - 1.3),
                      arrowprops=dict(arrowstyle='->', color='#aaaaaa', lw=1.5))

ax10.axis('off')

# ── (15) Separation quality across Dhātu axes ────────────────
ax11 = fig.add_subplot(4, 4, 15)
dark_ax(ax11, f"Inter-class Separation\nratio={separation_ratio:.2f}× "
              f"({'excellent' if separation_ratio>5 else 'good' if separation_ratio>2 else 'moderate'})",
        "Distance", "Count")
ax11.hist(intra_dists, bins=25, color='#ff6b6b', alpha=0.7, label='intra-class', density=True)
ax11.hist(inter_dists, bins=25, color='#6bcb77', alpha=0.7, label='inter-class', density=True)
ax11.legend(fontsize=6, facecolor='#222222', labelcolor='white')
ax11.axvline(mean_intra, color='#ff6b6b', ls='--', lw=1)
ax11.axvline(mean_inter, color='#6bcb77', ls='--', lw=1)

# ── (16) Training synapse curve ──────────────────────────────
ax12 = fig.add_subplot(4, 4, 16)
dark_ax(ax12, "Synapse pruning history", "Epoch", "Active synapses")
ax12.plot(synapse_history, color='#c77dff', lw=0.9)
ax12.axvline(200, color='yellow', lw=1, ls='--', alpha=0.7)
ax12.fill_between(range(len(synapse_history)), synapse_history,
                  alpha=0.2, color='#c77dff')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('ddin_v5_vaikhari.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d0d')
plt.show()
print("\nSaved: ddin_v5_vaikhari.png")


# ────────────────────────────────────────────────────────────────
# 11.  FINAL SUMMARY
# ────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("DDIN v5  —  VAIKHARĪ  —  FINAL SUMMARY")
print("="*60)

print(f"""
Network
  Sparsity  : {sparsity:.1%}
  Synapses  : {synapse_history[-1]}
  Dhātu     : {N_DHATU}

Coding quality
  Separation ratio : {separation_ratio:.2f}×   (>2.0=good, >5.0=excellent)
  Intra-class dist : {mean_intra:.4f}
  Inter-class dist : {mean_inter:.4f}

Vaikharī decoder
  Input dim   : {N_DHATU}  (Dhātu codes)
  Output dim  : {n_classes}  (semantic categories)
  Parameters  : {sum(p.numel() for p in decoder.parameters())}  (intentionally tiny)
  Val accuracy: {final_acc:.1%}  (chance={1/n_classes:.0%})

Pipeline proved
  raw input
    → DDIN dynamics (64 neurons, {synapse_history[-1]} synapses)
    → Dhātu activation code ({N_DHATU} numbers)
    → category label (1 of {n_classes})
""")

print("Complete DDIN Stack:")
print("  ✔  Parā      : physical substrate dynamics       (v1/v2)")
print("  ✔  Paśyantī  : coherent attractor states         (v3)")
print("  ✔  Madhyamā  : structured Dhātu subgraphs        (v4)")
print("  ✔  Vaikharī  : symbolic output                   (v5)  ← COMPLETE")

print("""
What this means:

  You now have the first end-to-end DDIN pipeline.
  Input enters the dynamical substrate.
  The Dhātu system structures the response.
  The Vaikharī head reads the structure and names it.

  The naming is done by a {param}–parameter decoder,
  not by the 64-neuron dynamical core.
  The SEMANTICS live in the dynamics, not the decoder.
  The decoder only reads out what the Dhātus already know.

Next frontier:
  The decoder is currently supervised (trained on mode labels we defined).
  True Vaikharī would be EMERGENT — the labels arise from usage patterns,
  not from our pre-defined categories.
  That is the Experiment 06 target: self-supervised Vaikharī.
""".format(param=sum(p.numel() for p in decoder.parameters())))

print("="*60)


Device: cpu
Training DDIN (same protocol as v4)...
  Epoch   0 (Learning) | loss=0.0904 | synapses=3489
  Epoch  50 (Learning) | loss=0.0658 | synapses=3495
  Epoch 100 (Learning) | loss=0.0606 | synapses=3495
  Epoch 150 (Learning) | loss=0.0591 | synapses=3503
  Epoch 200 (Pruning ) | loss=10.7300 | synapses=2890
  Epoch 250 (Pruning ) | loss=6.1097 | synapses=2230
  Epoch 300 (Pruning ) | loss=3.1018 | synapses=918
  Epoch 350 (Pruning ) | loss=1.6029 | synapses=467

Final sparsity: 94.1%   |   Synapses: 240

Dhātu cluster sizes: [11, 33, 1, 16, 3]

Computing Dhātu activation codes for all modalities...
Code matrix: (80, 5)  (samples × Dhātu)
Categories : [np.str_('ACCUMULATION'), np.str_('BURST'), np.str_('EVENT'), np.str_('FAST_WAVE'), np.str_('INVERSION'), np.str_('OSCILLATION'), np.str_('SLOW_WAVE'), np.str_('STEADY_STATE'), np.str_('STOCHASTIC')]

Separability:
  Mean intra-class distance : 0.0141
  Mean inter-class distance : 1.0262
  Separation ratio          : 72.62x
  (rati